### ĐỌC FILE

In [ ]:
import pandas as pd

df = pd.read_csv(r"D:\máy học\đồ án\code\data\RT_IOT2022.csv")
df.head()

In [ ]:
print(f"Số dòng: {df.shape[0]}")
print(f"Số cột: {df.shape[1]}")

- Nhận xét: 

### CÁC GIÁ TRỊ THIẾU, DỮ LIỆU TRÙNG VÀ OUTLIER

In [ ]:
# 1. Kiểm tra các giá trị bị thiếu
missing_values = df.isna().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

print(f"Tổng số giá trị bị thiếu: {df.isna().sum().sum()}")
if missing_values.empty:
    print("Không có giá trị bị thiếu.")
else:
    print("Số giá trị bị thiếu theo cột:")
    display(missing_values.to_frame(name="Số giá trị thiếu"))

# 2. Kiểm tra các dòng dữ liệu bị trùng
duplicate_count = df.duplicated().sum()
print(f"\nSố dòng bị trùng: {duplicate_count}")

# 3. Kiểm tra các giá trị âm trong các cột số
numeric_df = df.select_dtypes(include="number")
negative_values = (numeric_df < 0).sum()
negative_values = negative_values[negative_values > 0].sort_values(ascending=False)

print(f"\nTổng số giá trị âm trong các cột số: {negative_values.sum()}")
if negative_values.empty:
    print("Không phát hiện giá trị âm.")
else:
    print("Số giá trị âm theo cột:")
    display(negative_values.to_frame(name="Số giá trị âm"))

# 4. Phát hiện ngoại lệ bằng phương pháp IQR
q1 = numeric_df.quantile(0.25)
q3 = numeric_df.quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outlier_mask = (numeric_df < lower_bound) | (numeric_df > upper_bound)
outlier_counts = outlier_mask.sum()
outlier_counts = outlier_counts[outlier_counts > 0].sort_values(ascending=False)

print(f"\nTổng số giá trị ngoại lệ theo IQR: {outlier_counts.sum()}")
print("Số giá trị ngoại lệ theo từng cột:")
display(outlier_counts.to_frame(name="Số ngoại lệ"))


- Nhận xét:

### CÁC CỘT CÓ 1-2 GIÁ TRỊ

In [ ]:
# Tìm các cột chỉ có 1 hoặc 2 giá trị khác nhau
low_cardinality_columns = []

for column_name in df.columns:
    unique_count = df[column_name].nunique(dropna=False)
    if unique_count in (1, 2):
        low_cardinality_columns.append({
            "Tên cột": column_name,
            "Số giá trị khác nhau": unique_count
        })

low_cardinality_summary = pd.DataFrame(low_cardinality_columns)

if low_cardinality_summary.empty:
    print("Không có cột nào chỉ chứa 1 hoặc 2 giá trị khác nhau.")
else:
    print("Các cột chỉ có 1 hoặc 2 giá trị khác nhau:")
    display(low_cardinality_summary)

    # Hiển thị các giá trị và số lần xuất hiện của từng giá trị
    for column_name in low_cardinality_summary["Tên cột"]:
        print(f"\n{column_name}:")
        value_counts = df[column_name].value_counts(dropna=False)
        value_counts.index.name = "Giá trị"
        value_counts.name = "Số lần xuất hiện"
        display(value_counts.to_frame())

- Nhận xét: 

### Các dữ liệu có sự mất cân bằng hoặc có sự đột biến 

In [ ]:
import numpy as np

# Chỉ phân tích các cột số, bỏ cột số thứ tự
numeric_data = df.select_dtypes(include="number").drop(
    columns=["Unnamed: 0"], errors="ignore"
)

# Tính các thống kê cần thiết
column_statistics = pd.DataFrame({
    "Trung bình": numeric_data.mean(),
    "Trung vị": numeric_data.median(),
    "Độ lệch chuẩn": numeric_data.std(),
    "Độ lệch (skewness)": numeric_data.skew()
})

# Tính phần trăm chênh lệch giữa trung bình và trung vị
safe_mean = column_statistics["Trung bình"].abs().replace(0, np.nan)
column_statistics["Chênh lệch trung bình-trung vị (%)"] = (
    (column_statistics["Trung bình"] - column_statistics["Trung vị"]).abs()
    / safe_mean * 100
).fillna(0)

# Tìm giá trị đột biến bằng điểm Z: |z| > 3
standard_deviation = column_statistics["Độ lệch chuẩn"].replace(0, np.nan)
z_scores = (
    (numeric_data - column_statistics["Trung bình"]) / standard_deviation
).abs()
column_statistics["Số giá trị đột biến"] = (z_scores > 3).sum()
column_statistics["Tỷ lệ đột biến (%)"] = (
    column_statistics["Số giá trị đột biến"] / len(numeric_data) * 100
)

# In các cột có dữ liệu đột biến
print("Các cột có dữ liệu đột biến (|z| > 3):")
anomaly_columns = column_statistics[
    column_statistics["Số giá trị đột biến"] > 0
].sort_values("Số giá trị đột biến", ascending=False)

if anomaly_columns.empty:
    print("Không phát hiện dữ liệu đột biến.")
else:
    for column_name, row in anomaly_columns.iterrows():
        print(
            f"- {column_name}: {int(row['Số giá trị đột biến'])} giá trị "
            f"({row['Tỷ lệ đột biến (%)']:.2f}%)"
        )

# In các cột có dấu hiệu mất cân bằng hoặc lệch nhiều
print("\nCác cột có dấu hiệu mất cân bằng:")
imbalanced_columns = column_statistics[
    (column_statistics["Độ lệch (skewness)"].abs() > 1)
    | (column_statistics["Chênh lệch trung bình-trung vị (%)"] > 50)
].sort_values("Chênh lệch trung bình-trung vị (%)", ascending=False)

if imbalanced_columns.empty:
    print("Không phát hiện cột mất cân bằng rõ rệt.")
else:
    for column_name, row in imbalanced_columns.iterrows():
        print(
            f"- {column_name}: trung bình = {row['Trung bình']:.4f}, "
            f"trung vị = {row['Trung vị']:.4f}, "
            f"skewness = {row['Độ lệch (skewness)']:.4f}"
        )


- Nhận xét:

### Mutual information

- Xác định cột nhãn và cột thuộc tính

In [ ]:
# Xác định cột nhãn/ mục tiêu
label_column = "Attack_type"
target_column = label_column

# Kiểm tra cột tồn tại
if target_column not in df.columns:
    raise ValueError(
        f"Không tìm thấy cột mục tiêu '{target_column}'. "
        f"Các cột hiện có: {df.columns.tolist()}"
    )

# Tách dữ liệu đầu vào và nhãn
X = df.drop(columns=[target_column])
y = df[target_column]

print(f"Label column: {label_column}")
print(f"Target column: {target_column}")
print(f"Kích thước X: {X.shape}")
print(f"Kích thước y: {y.shape}")

- Mutual information

In [ ]:
from sklearn.feature_selection import mutual_info_classif

# Tên cột nhãn đã được xác định ở các cell trước
label_column = target_column

if label_column not in df.columns:
    raise ValueError(f"Không tìm thấy cột nhãn '{label_column}'.")

# Loại bỏ dòng thiếu nhãn
data_mi = df.dropna(subset=[label_column]).copy()

# Mã hóa nhãn mục tiêu
y_mi, _ = pd.factorize(data_mi[label_column].astype(str))

X_mi = []
feature_names = []
discrete_features = []

for column in data_mi.columns:
    if column == label_column:
        continue

    series = data_mi[column]

    if pd.api.types.is_numeric_dtype(series):
        values = pd.to_numeric(
            series, errors="coerce"
        ).replace([np.inf, -np.inf], np.nan)

        if values.notna().sum() == 0:
            continue

        values = values.fillna(values.median())
        X_mi.append(values.to_numpy())
        discrete_features.append(False)

    else:
        values = (
            series.astype("string")
            .fillna("__MISSING__")
            .factorize(sort=True)[0]
        )
        X_mi.append(values)
        discrete_features.append(True)

    feature_names.append(column)

if not X_mi:
    raise ValueError("Không có thuộc tính hợp lệ để tính Mutual Information.")

X_mi = np.column_stack(X_mi)

# Tính Mutual Information
mi_values = mutual_info_classif(
    X_mi,
    y_mi,
    discrete_features=discrete_features,
    random_state=42
)

# Sắp xếp thuộc tính theo mức độ quan trọng
mi_results = pd.DataFrame({
    "Thuộc tính": feature_names,
    "Mutual Information": mi_values
}).sort_values(
    by="Mutual Information",
    ascending=False
).reset_index(drop=True)

print("XẾP HẠNG THUỘC TÍNH THEO MUTUAL INFORMATION")
display(mi_results)

# Hiển thị 20 thuộc tính có MI cao nhất
top_n = min(20, len(mi_results))
top_features = mi_results.head(top_n)

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))
plt.barh(
    top_features["Thuộc tính"][::-1],
    top_features["Mutual Information"][::-1],
    color="steelblue",
    edgecolor="black"
)

plt.title(f"Top {top_n} thuộc tính theo Mutual Information")
plt.xlabel("Mutual Information")
plt.ylabel("Thuộc tính")
plt.tight_layout()
plt.show()

- Nhận xét: MutualInformatio cho biết biến liên quan đến payload, số packet, thời gian truyền và kích thước cửa sổ TCP có mức liên hệ cao với nhãn. fwd_pkts_payload.avg có giá trị cao nhất, vì vậy đây là nhóm đặc trưng tiềm năng cho mô hình.

### Sơ đồ boxplot của top5 dữ liệu có giá trị bất thường nhiều nhất và có điểm mutual information cao nhất

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt

# Lấy top 5 cột có số lượng ngoại lệ nhiều nhất
boxplot_columns = outlier_counts.head(5).index.tolist()

# Tạo folder lưu ảnh
output_folder = Path("boxplot_images")
output_folder.mkdir(parents=True, exist_ok=True)

valid_columns = [
    column_name for column_name in boxplot_columns
    if column_name in df.columns
]

for column_name in valid_columns:
    figure, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Boxplot dữ liệu gốc
    df[[column_name]].boxplot(
        ax=axes[0],
        showfliers=False,
        grid=False
    )
    axes[0].set_title(f"{column_name} - Dữ liệu gốc")
    axes[0].set_ylabel("Giá trị")

    # Boxplot sau biến đổi log1p
    log_data = np.log1p(df[[column_name]])
    log_data.boxplot(
        ax=axes[1],
        showfliers=True,
        grid=False
    )
    axes[1].set_title(f"{column_name} - Sau log1p")
    axes[1].set_ylabel("log(1 + giá trị)")

    figure.suptitle(
        f"Phân phối thuộc tính: {column_name}",
        fontsize=14
    )
    plt.tight_layout()

    # Làm sạch tên file
    safe_name = re.sub(r'[\\/*?:"<>|]', "_", column_name)
    image_path = output_folder / f"boxplot_{safe_name}.png"

    # Lưu ảnh
    figure.savefig(image_path, dpi=300, bbox_inches="tight")
    print(f"Đã lưu: {image_path}")

    plt.show()
    plt.close(figure)


- Nhận xét: 
    - boxplot_flow_pkts_payload.avg: Dữ liệu gốc bị nén mạnh quanh một vùng giá trị; sau log1p xuất hiện nhiều điểm ngoại lệ, cho thấy phân bố không đồng đều.
    - boxplot_flow_ACK_flag_count: Giá trị trung tâm gần 1, nhưng có nhiều ngoại lệ lớn sau biến đổi log.
    - boxplot_bwd_header_size_max: Dữ liệu gốc gần như tập trung tại giá trị 20. Sau log1p, các giá trị 0, 8, 32 và 40 được thể hiện rõ hơn.
    - boxplot_bwd_header_size_tot: Phần lớn giá trị nhỏ, trong khi nhiều điểm nằm xa trung tâm sau log, chứng tỏ có nhiều ngoại lệ lớn.
    - boxplot_bwd_pkts_tot: Số packet chiều ngược chủ yếu thấp, nhưng tồn tại một số phiên có giá trị rất cao.

### Sơ đồ histogram của top5 dữ liệu có giá trị bất thường nhiều nhất 

In [ ]:
# Lấy 5 cột có nhiều giá trị ngoại lệ nhất
histogram_columns = outlier_counts.head(5).index.tolist()

# Tạo thư mục lưu ảnh histogram
histogram_folder = Path("histogram_images")
histogram_folder.mkdir(parents=True, exist_ok=True)

for column_name in histogram_columns:
    values = pd.to_numeric(df[column_name], errors="coerce")
    values = values.replace([np.inf, -np.inf], np.nan).dropna()

    if values.empty:
        continue

    # Giới hạn vùng trung tâm theo phân vị
    lower_percentile = values.quantile(0.01)
    upper_percentile = values.quantile(0.99)
    central_values = values[
        (values >= lower_percentile) & (values <= upper_percentile)
    ]

    # Nén giá trị lớn bằng log1p
    if values.min() >= 0:
        compressed_values = np.log1p(values)
        compressed_label = "log(1 + giá trị)"
    else:
        compressed_values = np.sign(values) * np.log1p(values.abs())
        compressed_label = "signed log1p"

    figure, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.ravel()

    # 1. Histogram dữ liệu gốc
    axes[0].hist(
        values,
        bins=50,
        color="steelblue",
        edgecolor="black"
    )
    axes[0].set_title("1. Dữ liệu gốc")
    axes[0].set_xlabel("Giá trị")
    axes[0].set_ylabel("Tần suất")

    # 2. Nén các giá trị lớn
    axes[1].hist(
        compressed_values,
        bins=50,
        color="darkorange",
        edgecolor="black"
    )
    axes[1].set_title("2. Nén giá trị lớn")
    axes[1].set_xlabel(compressed_label)
    axes[1].set_ylabel("Tần suất")

    # 3. Nén giá trị và nén chênh lệch số lượng
    axes[2].hist(
        compressed_values,
        bins=50,
        color="seagreen",
        edgecolor="black",
        log=True
    )
    axes[2].set_title("3. Nén giá trị và tần suất")
    axes[2].set_xlabel(compressed_label)
    axes[2].set_ylabel("Tần suất (log scale)")

    # 4. Chỉ hiển thị vùng trung tâm
    axes[3].hist(
        central_values,
        bins=50,
        color="mediumpurple",
        edgecolor="black"
    )
    axes[3].set_title("4. Vùng trung tâm (1% - 99%)")
    axes[3].set_xlabel("Giá trị")
    axes[3].set_ylabel("Tần suất")

    for axis in axes:
        axis.grid(axis="y", alpha=0.3)

    figure.suptitle(
        f"Histogram của thuộc tính: {column_name}",
        fontsize=16
    )
    figure.tight_layout()

    safe_name = re.sub(r'[\\/*?:"<>|]', "_", column_name)
    image_path = histogram_folder / f"histogram_{safe_name}.png"

    figure.savefig(image_path, dpi=300, bbox_inches="tight")
    print(f"Đã lưu: {image_path}")

    plt.show()
    plt.close(figure)

- Nhận xét:
    - histogram_bwd_header_size_max: Dữ liệu tập trung tại một số giá trị rời rạc, đặc biệt quanh 0, 8, 20 và 40. Biến đổi log không làm thay đổi bản chất rời rạc nhưng giúp quan sát rõ hơn.
    - histogram_bwd_header_size_tot: Phân bố lệch phải mạnh, phần lớn giá trị nhỏ nhưng có một số giá trị rất lớn. log1p giúp giảm ảnh hưởng của các giá trị cực đoan.
    - histogram_bwd_pkts_tot: Phần lớn số packet chiều ngược nhỏ, chỉ một số ít phiên có số packet rất lớn. Đây là biến có nhiều ngoại lệ.
    - histogram_flow_ACK_flag_count: Giá trị chủ yếu tập trung ở 0 và 1; các giá trị cao hơn giảm nhanh và tạo đuôi phải dài.
    - histogram_flow_pkts_payload.avg: Giá trị tập trung tại một vài vùng, đặc biệt khoảng 60–100 và 200–300. Phân bố có độ phân tán lớn, nên biến đổi log là cần thiết.

### Pairplot và countplot

In [ ]:
import seaborn as sns

# Thư mục lưu biểu đồ
pairplot_folder = Path("pairplot_images")
countplot_folder = Path("countplot_images")
pairplot_folder.mkdir(exist_ok=True)
countplot_folder.mkdir(exist_ok=True)

# Các nhóm thuộc tính cho nhiều pairplot
mi_columns = top_features["Thuộc tính"].tolist()
outlier_columns = outlier_counts.head(9).index.tolist()

pairplot_groups = []

for columns in [mi_columns, outlier_columns]:
    for start in range(0, len(columns), 3):
        group = columns[start:start + 3]
        if len(group) == 3:
            pairplot_groups.append(group)

# Vẽ nhiều pairplot
for index, columns in enumerate(pairplot_groups, start=1):
    current_data = df[columns + [target_column]].copy()
    current_data[target_column] = current_data[target_column].astype(str)
    current_data = current_data.sample(
        n=min(5000, len(current_data)),
        random_state=42
    )

    grid = sns.pairplot(
        current_data,
        vars=columns,
        hue=target_column,
        palette="tab20",
        diag_kind="hist",
        corner=True,
        plot_kws={
            "alpha": 0.65,
            "s": 20,
            "edgecolor": "black",
            "linewidth": 0.2
        }
    )

    grid.fig.suptitle(
        f"Pairplot nhóm {index} theo {target_column}",
        y=1.02
    )

    image_path = pairplot_folder / f"pairplot_group_{index}.png"
    grid.fig.savefig(image_path, dpi=300, bbox_inches="tight")
    print(f"Đã lưu: {image_path}")
    plt.show()
    plt.close(grid.fig)

# Các cột phù hợp để vẽ countplot
countplot_columns = [target_column]

for column_name in df.columns:
    if column_name != target_column and df[column_name].nunique() <= 20:
        countplot_columns.append(column_name)

countplot_columns = countplot_columns[:10]

# Vẽ nhiều countplot
for index, column_name in enumerate(countplot_columns, start=1):
    order = df[column_name].value_counts().index

    figure, axis = plt.subplots(figsize=(12, 6))

    sns.countplot(
        data=df,
        x=column_name,
        order=order,
        color="steelblue",
        ax=axis
    )

    axis.set_title(f"Countplot của {column_name}")
    axis.set_xlabel(column_name)
    axis.set_ylabel("Số lượng")
    axis.tick_params(axis="x", rotation=45)

    for container in axis.containers:
        axis.bar_label(container, fmt="%d", padding=3, fontsize=8)

    figure.tight_layout()

    safe_name = re.sub(r'[\\/*?:"<>|]', "_", column_name)
    image_path = countplot_folder / f"countplot_{safe_name}.png"

    figure.savefig(image_path, dpi=300, bbox_inches="tight")
    print(f"Đã lưu: {image_path}")
    plt.show()
    plt.close(figure)


- Nhận xét countplot: 
    - countplot_Attack_type: Khẳng định sự chênh lệch lớn giữa các lớp. Mô hình có thể dự đoán tốt lớp lớn nhưng bỏ sót các lớp hiếm.
    - countplot`_proto: TCP xuất hiện nhiều nhất, UDP đứng thứ hai, ICMP rất ít. Dữ liệu chủ yếu mô tả các luồng TCP.
    - countplot_service: Phần lớn bản ghi không xác định dịch vụ (-). DNS, MQTT, HTTP và SSL là các dịch vụ phổ biến nhất.
    - countplot_flow_SYN_flag_count: Giá trị 1 chiếm ưu thế, tiếp theo là 0 và 2. Các giá trị lớn hơn 2 rất hiếm.
    - countplot_flow_FIN_flag_count: Hầu hết bằng 0, cho thấy phần lớn luồng không có FIN hoặc chỉ có rất ít FIN.
    - countplot_flow_RST_flag_count: Giá trị 1 và 0 chiếm chủ yếu. Một số ít luồng có nhiều RST, có thể liên quan đến hoạt động quét hoặc kết nối bất thường.
    - countplot_fwd_header_size_min: Giá trị 20 và 32 xuất hiện nhiều nhất, phản ánh các kích thước header TCP phổ biến.
    - countplot_fwd_header_size_max: Giá trị 20 chiếm đa số; các giá trị 8 và 40 cũng xuất hiện đáng kể.
    - countplot_bwd_header_size_min: Giá trị 20 chiếm nhiều nhất, sau đó là 0, 32 và 8.
    - countplot_bwd_header_size_max: Tập trung chủ yếu tại 20, 0, 40 và 8; các giá trị khác xuất hiện rất ít.

- Nhận xét pairplot:
    - pairplot_group_1: Các biến payload và số byte có quan hệ dương; ARP_poisioning phân bố ở vùng giá trị cao hơn, trong khi các lớp khác tập trung gần gốc.
    - pairplot_group_2: Các biến tổng payload có tương quan rõ. ARP_poisioning có nhiều giá trị lớn và tách khỏi phần lớn lớp còn lại.
    - pairplot_group_3: id.resp_p và kích thước cửa sổ có khả năng phân biệt một số lớp NMAP. Dữ liệu phân bố không đồng đều và có nhiều giá trị cực lớn.
    - pairplot_group_4: Các biến kích thước payload và header có tương quan; ARP_poisioning thường tạo ra các điểm ở vùng cao.
    - pairplot_group_5: Các biến IAT có quan hệ dương rõ rệt. Một số lớp có độ trễ rất lớn, giúp phân biệt chúng với các luồng thông thường.
    - pairplot_group_6: active.tot và active.max tương quan mạnh. Phần lớn dữ liệu có giá trị thấp, chỉ một số ít phiên có thời gian hoạt động lớn.
    - pairplot_group_7: Kích thước header chiều ngược phần lớn nhỏ; ARP_poisioning có nhiều điểm lớn và phân tán rộng.
    - pairplot_group_8: Số ACK, số packet chiều ngược và số packet trong subflow có tương quan dương. Các điểm lớn chủ yếu thuộc một số lớp tấn công nhất định.
    - pairplot_group_9: fwd_pkts_payload.avg phân tách tương đối tốt một số nhóm, nhưng các lớp vẫn chồng lấn đáng kể. Cần kết hợp nhiều biến khi phân loại.

### Phân tích nhãn

In [ ]:
# Phân bố dữ liệu theo nhãn Attack_type
label_counts = df[target_column].value_counts()
label_percentages = label_counts.div(label_counts.sum()).mul(100)

label_distribution = pd.DataFrame({
    "Số lượng": label_counts,
    "Tỷ lệ (%)": label_percentages.round(2)
})

display(label_distribution)

# Vẽ biểu đồ phân bố nhãn
figure, axis = plt.subplots(figsize=(14, 7))

bars = sns.barplot(
    x=label_counts.values,
    y=label_counts.index,
    order=label_counts.index,
    color="steelblue",
    ax=axis
)

axis.set_title("Phân bố dữ liệu theo nhãn tấn công", fontsize=16)
axis.set_xlabel("Số lượng mẫu")
axis.set_ylabel("Nhãn")

for bar, percentage in zip(bars.patches, label_percentages):
    axis.text(
        bar.get_width() + label_counts.max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{int(bar.get_width()):,} ({percentage:.2f}%)",
        va="center",
        fontsize=9
    )

axis.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

# Đánh giá mức độ mất cân bằng
imbalance_ratio = label_counts.max() / label_counts.min()

print(f"Nhãn có nhiều mẫu nhất: {label_counts.idxmax()} ({label_counts.max():,} mẫu)")
print(f"Nhãn có ít mẫu nhất: {label_counts.idxmin()} ({label_counts.min():,} mẫu)")
print(f"Tỷ lệ mất cân bằng lớn nhất/nhỏ nhất: {imbalance_ratio:.2f}")



- Nhận xét: DOS_SYN_Hping chiếm áp đảo với 76,89%. Các lớp như NMAP_FIN_SCAN, Metasploit_Brute_Force_SSH có rất ít mẫu, cho thấy dữ liệu mất cân bằng nghiêm trọng.

### Nhận xét chung: Các biểu đồ cho thấy dữ liệu có nhiều đặc trưng phân biệt tốt, nhưng bị lệch phải, nhiều ngoại lệ và mất cân bằng lớp. Vì vậy cần xử lý dữ liệu trước khi huấn luyện mô hình.